**MySQL Schema Creation & Business Queries**

WHAT WAS BUILT
* Database : ecommerce_analytics (MySQL)
 
  * Tables created (Star Schema):
    1) dim_customers       — customer dimension
    2) dim_products        — product dimension
    3) dim_dates           — date dimension
    4) fact_transactions   — core sales fact table
    5) fact_cancellations  — returns fact table
 
  * Business Queries Run:
    1)  Revenue & orders by country (window function)
    2)  Top 10 customers by revenue
    3)  Revenue share by product category
    4)  Monthly trend with MoM change (LAG)
    5)  Cancellation rate by category (subquery + JOIN)
    6)  Customer loyalty segmentation (CTE + CASE WHEN)
    7)  Best-selling products (excl. bulk)
    8)  Weekend vs weekday revenue (JOIN + window)

In [2]:
import pandas as pd
import numpy as np
import mysql.connector
from mysql.connector import Error
import warnings
warnings.filterwarnings('ignore')

**MYSQL CREDENTIALS**

In [3]:
DB_HOST     = "localhost"            # usually localhost
DB_PORT     = 3306                   # default MySQL port
DB_USER     = "root"                 # ← your MySQL username
DB_PASSWORD = "Vidhi@1155"        # ← your MySQL password
DB_NAME     = "ecommerce_analytics"  # database you created

**HELPER: run a query and return results as a DataFrame**

In [4]:
def run_query(cursor, sql, title=None):
    cursor.execute(sql)
    rows    = cursor.fetchall()
    columns = [d[0] for d in cursor.description]
    df      = pd.DataFrame(rows, columns=columns)
    if title:
        print(f"\n{'═'*60}")
        print(f"  {title}")
        print(f"{'═'*60}")
        print(df.to_string(index=False))
    return df

**1. CONNECT to Database**

In [5]:
print("\n[CONNECT] Connecting to MySQL...")
try:
    conn = mysql.connector.connect(
        host     = DB_HOST,
        port     = DB_PORT,
        user     = DB_USER,
        password = DB_PASSWORD,
        database = DB_NAME,
        charset  = 'utf8mb4'
    )
    cursor = conn.cursor()
    print(f"Connected to '{DB_NAME}' on {DB_HOST}:{DB_PORT}")
except Error as e:
    print(f"Connection failed: {e}")
    print("\n  Checklist:")
    print("    1. Is MySQL running?")
    print("    2. Are DB_USER and DB_PASSWORD correct?")
    print("    3. Did you create the database?")
    print("       → Run in MySQL Workbench: CREATE DATABASE ecommerce_analytics;")
    exit(1)


[CONNECT] Connecting to MySQL...
Connected to 'ecommerce_analytics' on localhost:3306


**2. LOAD CSVs**

In [6]:
print("\n[LOAD] Reading CSV files...")
df       = pd.read_csv('data/online_retail_cleaned.csv', parse_dates=['InvoiceDate'])
df_cat   = pd.read_csv('data/product_catalog.csv')
df_canc  = pd.read_csv('data/cancellations.csv')
 
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['StockCode']   = df['StockCode'].astype(str)
df['CustomerID']  = df['CustomerID'].astype(str)
df['IsBulkOrder'] = df['IsBulkOrder'].astype(int)   # bool → 0/1 for MySQL
df['IsWeekend']   = df['IsWeekend'].astype(int)
 
df_cat['StockCode'] = df_cat['StockCode'].astype(str)
 
print(f"  Transactions  : {len(df):,} rows")
print(f"  Products      : {len(df_cat):,} rows")
print(f"  Cancellations : {len(df_canc):,} rows")


[LOAD] Reading CSV files...
  Transactions  : 89,827 rows
  Products      : 93 rows
  Cancellations : 1,821 rows


**3. DROP OLD TABLES (clean slate)**

In [8]:
print("\n[SCHEMA] Dropping existing tables (if any)...")
cursor.execute("SET FOREIGN_KEY_CHECKS = 0;")
for tbl in ['fact_transactions', 'dim_customers',
            'dim_products', 'dim_dates', 'fact_cancellations']:
    cursor.execute(f"DROP TABLE IF EXISTS {tbl};")
    print(f"  Dropped (if existed): {tbl}")
cursor.execute("SET FOREIGN_KEY_CHECKS = 1;")
conn.commit()


[SCHEMA] Dropping existing tables (if any)...
  Dropped (if existed): fact_transactions
  Dropped (if existed): dim_customers
  Dropped (if existed): dim_products
  Dropped (if existed): dim_dates
  Dropped (if existed): fact_cancellations


**4. CREATE TABLES  — Star Schema**

In [9]:
print("\n[SCHEMA] Creating star schema tables...")
 
# ── dim_customers ──────────────────────────────────────────
cursor.execute("""
CREATE TABLE dim_customers (
    CustomerID    VARCHAR(20)    NOT NULL,
    Country       VARCHAR(60)    NOT NULL,
    TotalOrders   INT            DEFAULT 0,
    TotalRevenue  DECIMAL(12,2)  DEFAULT 0.00,
    AvgOrderValue DECIMAL(10,2)  DEFAULT 0.00,
    IsGuest       TINYINT(1)     DEFAULT 0,
    PRIMARY KEY (CustomerID)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
""")
print("dim_customers")
 
# ── dim_products ───────────────────────────────────────────
cursor.execute("""
CREATE TABLE dim_products (
    StockCode    VARCHAR(20)   NOT NULL,
    Description  VARCHAR(255),
    Category     VARCHAR(60),
    AvgUnitPrice DECIMAL(10,2) DEFAULT 0.00,
    PRIMARY KEY (StockCode)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
""")
print("dim_products")
 
# ── dim_dates ──────────────────────────────────────────────
cursor.execute("""
CREATE TABLE dim_dates (
    InvoiceDate  DATE        NOT NULL,
    Year         SMALLINT,
    Month        TINYINT,
    MonthName    VARCHAR(10),
    Quarter      TINYINT,
    DayOfWeek    VARCHAR(15),
    IsWeekend    TINYINT(1),
    PRIMARY KEY (InvoiceDate)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
""")
print("dim_dates")
 
# ── fact_transactions ──────────────────────────────────────
cursor.execute("""
CREATE TABLE fact_transactions (
    TransactionID INT           NOT NULL AUTO_INCREMENT,
    InvoiceNo     VARCHAR(20)   NOT NULL,
    StockCode     VARCHAR(20)   NOT NULL,
    CustomerID    VARCHAR(20)   NOT NULL,
    InvoiceDate   DATE          NOT NULL,
    Quantity      INT,
    UnitPrice     DECIMAL(10,2),
    Revenue       DECIMAL(12,2),
    Country       VARCHAR(60),
    Category      VARCHAR(60),
    YearMonth     VARCHAR(10),
    IsBulkOrder   TINYINT(1)    DEFAULT 0,
    PRIMARY KEY (TransactionID),
    FOREIGN KEY (StockCode)   REFERENCES dim_products(StockCode),
    FOREIGN KEY (CustomerID)  REFERENCES dim_customers(CustomerID),
    FOREIGN KEY (InvoiceDate) REFERENCES dim_dates(InvoiceDate)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
""")
print("fact_transactions")
 
# ── fact_cancellations ─────────────────────────────────────
cursor.execute("""
CREATE TABLE fact_cancellations (
    CancelID     INT          NOT NULL AUTO_INCREMENT,
    InvoiceNo    VARCHAR(20),
    StockCode    VARCHAR(20),
    Description  VARCHAR(255),
    Quantity     INT,
    InvoiceDate  VARCHAR(30),
    UnitPrice    DECIMAL(10,2),
    CustomerID   VARCHAR(20),
    Country      VARCHAR(60),
    Category     VARCHAR(60),
    PRIMARY KEY (CancelID)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
""")
print("fact_cancellations")
conn.commit()


[SCHEMA] Creating star schema tables...
dim_customers
dim_products
dim_dates
fact_transactions
fact_cancellations


**5. POPULATE DIMENSION TABLES**

In [10]:
print("\n[INSERT] Populating dimension tables...")
 
# ── dim_customers ──────────────────────────────────────────
cust_agg = (
    df.groupby('CustomerID')
    .agg(
        Country       = ('Country',   'first'),
        TotalOrders   = ('InvoiceNo', 'nunique'),
        TotalRevenue  = ('Revenue',   'sum'),
    )
    .reset_index()
)
cust_agg['AvgOrderValue'] = (cust_agg['TotalRevenue'] / cust_agg['TotalOrders']).round(2)
cust_agg['IsGuest']       = (cust_agg['CustomerID'] == 'Guest').astype(int)
cust_agg['TotalRevenue']  = cust_agg['TotalRevenue'].round(2)
 
cust_rows = [tuple(r) for r in cust_agg[
    ['CustomerID','Country','TotalOrders','TotalRevenue','AvgOrderValue','IsGuest']
].itertuples(index=False)]
 
cursor.executemany("""
    INSERT INTO dim_customers
        (CustomerID, Country, TotalOrders, TotalRevenue, AvgOrderValue, IsGuest)
    VALUES (%s, %s, %s, %s, %s, %s)
""", cust_rows)
conn.commit()
print(f"dim_customers   : {len(cust_rows):,} records")
 
# ── dim_products ───────────────────────────────────────────
prod_agg = (
    df.groupby('StockCode')
    .agg(
        Description  = ('Description', 'first'),
        Category     = ('Category',    'first'),
        AvgUnitPrice = ('UnitPrice',   'mean'),
    )
    .reset_index()
)
prod_agg['AvgUnitPrice'] = prod_agg['AvgUnitPrice'].round(2)
prod_agg['Description']  = prod_agg['Description'].fillna('Unknown Product')
 
prod_rows = [tuple(r) for r in prod_agg[
    ['StockCode','Description','Category','AvgUnitPrice']
].itertuples(index=False)]
 
cursor.executemany("""
    INSERT INTO dim_products (StockCode, Description, Category, AvgUnitPrice)
    VALUES (%s, %s, %s, %s)
""", prod_rows)
conn.commit()
print(f"dim_products    : {len(prod_rows):,} records")
 
# ── dim_dates ──────────────────────────────────────────────
date_df = df[['InvoiceDate','Year','Month','MonthName',
              'Quarter','DayOfWeek','IsWeekend']].drop_duplicates(subset='InvoiceDate')
date_df['InvoiceDate'] = date_df['InvoiceDate'].dt.date
 
date_rows = [tuple(r) for r in date_df.itertuples(index=False)]
cursor.executemany("""
    INSERT IGNORE INTO dim_dates
        (InvoiceDate, Year, Month, MonthName, Quarter, DayOfWeek, IsWeekend)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
""", date_rows)
conn.commit()
print(f"dim_dates       : {len(date_rows):,} records")


[INSERT] Populating dimension tables...
dim_customers   : 4,696 records
dim_products    : 93 records
dim_dates       : 731 records


**6. POPULATE FACT TABLES  (batch insert for speed)**

In [11]:
print("\n[INSERT] Loading fact tables (batch insert)...")
 
# ── fact_transactions ──────────────────────────────────────
fact_cols = ['InvoiceNo','StockCode','CustomerID','InvoiceDate',
             'Quantity','UnitPrice','Revenue','Country',
             'Category','YearMonth','IsBulkOrder']
 
fact_df = df[fact_cols].copy()
fact_df['InvoiceDate'] = fact_df['InvoiceDate'].dt.date
fact_df['Revenue']     = fact_df['Revenue'].round(2)
fact_df['UnitPrice']   = fact_df['UnitPrice'].round(2)
fact_df['InvoiceNo']   = fact_df['InvoiceNo'].astype(str)
 
BATCH = 5000
total = len(fact_df)
inserted = 0
for i in range(0, total, BATCH):
    batch = fact_df.iloc[i:i+BATCH]
    rows  = [tuple(r) for r in batch.itertuples(index=False)]
    cursor.executemany("""
        INSERT INTO fact_transactions
            (InvoiceNo, StockCode, CustomerID, InvoiceDate,
             Quantity, UnitPrice, Revenue, Country,
             Category, YearMonth, IsBulkOrder)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, rows)
    conn.commit()
    inserted += len(rows)
    print(f"  fact_transactions : {inserted:,}/{total:,} rows inserted...", end='\r')
 
print(f"\nfact_transactions: {inserted:,} records")
 
# ── fact_cancellations ─────────────────────────────────────
canc_cols = ['InvoiceNo','StockCode','Description','Quantity',
             'InvoiceDate','UnitPrice','CustomerID','Country','Category']
 
df_canc_clean = df_canc.copy()
df_canc_clean['InvoiceNo']  = df_canc_clean['InvoiceNo'].astype(str)
df_canc_clean['StockCode']  = df_canc_clean['StockCode'].astype(str)
df_canc_clean['CustomerID'] = df_canc_clean['CustomerID'].fillna('Guest').astype(str)
df_canc_clean['UnitPrice']  = df_canc_clean['UnitPrice'].round(2)
df_canc_clean['Description']= df_canc_clean['Description'].fillna('Unknown')
 
canc_rows = [tuple(r) for r in df_canc_clean[canc_cols].itertuples(index=False)]
cursor.executemany("""
    INSERT INTO fact_cancellations
        (InvoiceNo, StockCode, Description, Quantity,
         InvoiceDate, UnitPrice, CustomerID, Country, Category)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
""", canc_rows)
conn.commit()
print(f"fact_cancellations: {len(canc_rows):,} records")


[INSERT] Loading fact tables (batch insert)...
  fact_transactions : 89,827/89,827 rows inserted...
fact_transactions: 89,827 records
fact_cancellations: 1,821 records


**7. VERIFY ROW COUNTS**

In [12]:
print("\n[VERIFY] Table row counts:")
for tbl in ['dim_customers','dim_products','dim_dates',
            'fact_transactions','fact_cancellations']:
    cursor.execute(f"SELECT COUNT(*) FROM {tbl}")
    count = cursor.fetchone()[0]
    print(f"  {tbl:<25}: {count:>8,} rows")


[VERIFY] Table row counts:
  dim_customers            :    4,696 rows
  dim_products             :       93 rows
  dim_dates                :      731 rows
  fact_transactions        :   89,827 rows
  fact_cancellations       :    1,821 rows


**8. BUSINESS QUERIES**

In [23]:
conn = mysql.connector.connect(
    host     = DB_HOST,
    port     = DB_PORT,
    user     = DB_USER,
    password = DB_PASSWORD,
    database = DB_NAME,
    charset  = 'utf8mb4'
)
cursor = conn.cursor()
print("Reconnected successfully")

Reconnected successfully


In [29]:
import os
os.chdir(r'C:\Users\HP\E-commerce Analytics')
print("Working directory set to:", os.getcwd())
os.makedirs('sql_queries_outputs', exist_ok=True)
print("  Results saved to: sql_queries_outputs/")

Working directory set to: C:\Users\HP\E-commerce Analytics
  Results saved to: sql_queries_outputs/


In [33]:
# Excel writer — all queries as separate sheets in one file
excel_path   = 'sql_queries_output/all_query_results.xlsx'
excel_writer = pd.ExcelWriter(excel_path, engine='openpyxl')
 
# ─────────────────────────────────────────────────────────
# Q1 | Revenue by Country — RANK() + SUM() OVER() + running total
# Window functions: RANK(), SUM() OVER(), cumulative SUM() OVER(ORDER BY)
# ─────────────────────────────────────────────────────────
q1 = run_query(cursor, """
    SELECT
        Country,
        COUNT(DISTINCT InvoiceNo)                        AS TotalOrders,
        ROUND(SUM(Revenue), 2)                           AS TotalRevenue,
        RANK() OVER (ORDER BY SUM(Revenue) DESC)         AS RevenueRank,
        ROUND(SUM(Revenue) * 100.0
              / SUM(SUM(Revenue)) OVER (), 2)            AS RevenueShare_Pct,
        ROUND(SUM(SUM(Revenue)) OVER (
              ORDER BY SUM(Revenue) DESC
              ROWS BETWEEN UNBOUNDED PRECEDING
              AND CURRENT ROW), 2)                       AS CumulativeRevenue
    FROM fact_transactions
    GROUP BY Country
    ORDER BY RevenueRank;
""", "Q1 | Revenue by Country — RANK() + Cumulative Revenue")
q1.to_csv('sql_queries_output/q1_revenue_by_country.csv', index=False)
q1.to_excel(excel_writer, sheet_name='Q1_Country_Revenue', index=False)
print("  → Window: RANK() ranks countries; cumulative SUM() OVER(ORDER BY) "
      "shows how quickly top countries account for total revenue.")
print("  → Saved: q1_revenue_by_country.csv\n")
 
 
# ─────────────────────────────────────────────────────────
# Q2 | Customer Revenue Ranking — DENSE_RANK() + NTILE()
# Window functions: DENSE_RANK(), NTILE(10) for decile bucketing
# ─────────────────────────────────────────────────────────
q2 = run_query(cursor, """
    WITH customer_revenue AS (
        SELECT
            CustomerID,
            COUNT(DISTINCT InvoiceNo)   AS TotalOrders,
            ROUND(SUM(Revenue), 2)      AS TotalRevenue,
            ROUND(AVG(Revenue), 2)      AS AvgOrderValue
        FROM fact_transactions
        WHERE CustomerID != 'Guest'
          AND IsBulkOrder = 0
        GROUP BY CustomerID
    )
    SELECT
        CustomerID,
        TotalOrders,
        TotalRevenue,
        AvgOrderValue,
        DENSE_RANK() OVER (ORDER BY TotalRevenue DESC)  AS RevenueRank,
        NTILE(10)    OVER (ORDER BY TotalRevenue DESC)  AS RevenueDecile
    FROM customer_revenue
    ORDER BY RevenueRank
    LIMIT 20;
""", "Q2 | Customer Revenue Ranking — DENSE_RANK() + NTILE(10)")
q2.to_csv('sql_queries_output/q2_customer_ranking.csv', index=False)
q2.to_excel(excel_writer, sheet_name='Q2_Customer_Ranking', index=False)
print("  → Window: DENSE_RANK() gives tied customers the same rank; "
      "NTILE(10) buckets all customers into revenue deciles (1=top 10%).")
print("  → Saved: q2_customer_ranking.csv\n")
 
 
# ─────────────────────────────────────────────────────────
# Q3 | Top 3 Products per Category — ROW_NUMBER() PARTITION BY
# Window function: ROW_NUMBER() OVER(PARTITION BY Category ORDER BY Revenue)
# ─────────────────────────────────────────────────────────
q3 = run_query(cursor, """
    WITH product_revenue AS (
        SELECT
            ft.StockCode,
            dp.Description,
            ft.Category,
            SUM(ft.Quantity)             AS UnitsSold,
            ROUND(SUM(ft.Revenue), 2)    AS TotalRevenue,
            COUNT(DISTINCT ft.InvoiceNo) AS TotalOrders
        FROM fact_transactions ft
        JOIN dim_products dp ON ft.StockCode = dp.StockCode
        WHERE ft.IsBulkOrder = 0
        GROUP BY ft.StockCode, dp.Description, ft.Category
    ),
    ranked AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY Category
                ORDER BY TotalRevenue DESC
            ) AS RankInCategory
        FROM product_revenue
    )
    SELECT
        Category,
        RankInCategory,
        StockCode,
        Description,
        UnitsSold,
        TotalRevenue,
        TotalOrders
    FROM ranked
    WHERE RankInCategory <= 3
    ORDER BY Category, RankInCategory;
""", "Q3 | Top 3 Products per Category — ROW_NUMBER() PARTITION BY")
q3.to_csv('sql_queries_output/q3_top_products_per_category.csv', index=False)
q3.to_excel(excel_writer, sheet_name='Q3_Top_Products_Category', index=False)
print("  → Window: ROW_NUMBER() PARTITION BY Category restarts numbering "
      "for each category — gives top-N per group without complex subqueries.")
print("  → Saved: q3_top_products_per_category.csv\n")
 
 
# ─────────────────────────────────────────────────────────
# Q4 | Monthly Trend — LAG() + LEAD() + Running Total
# Window functions: LAG(), LEAD(), SUM() OVER(ORDER BY) running total
# ─────────────────────────────────────────────────────────
q4 = run_query(cursor, """
    WITH monthly AS (
        SELECT
            YearMonth,
            COUNT(DISTINCT InvoiceNo)  AS TotalOrders,
            ROUND(SUM(Revenue), 2)     AS MonthlyRevenue
        FROM fact_transactions
        GROUP BY YearMonth
    )
    SELECT
        YearMonth,
        TotalOrders,
        MonthlyRevenue,
        LAG(MonthlyRevenue)  OVER (ORDER BY YearMonth)  AS PrevMonthRevenue,
        LEAD(MonthlyRevenue) OVER (ORDER BY YearMonth)  AS NextMonthRevenue,
        ROUND(MonthlyRevenue
              - LAG(MonthlyRevenue) OVER (ORDER BY YearMonth), 2)
                                                         AS MoM_Change,
        ROUND((MonthlyRevenue
               - LAG(MonthlyRevenue) OVER (ORDER BY YearMonth))
              * 100.0
              / NULLIF(LAG(MonthlyRevenue) OVER (ORDER BY YearMonth), 0),
              2)                                         AS MoM_Change_Pct,
        ROUND(SUM(MonthlyRevenue) OVER (
              ORDER BY YearMonth
              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2)
                                                         AS CumulativeRevenue
    FROM monthly
    ORDER BY YearMonth;
""", "Q4 | Monthly Trend — LAG() + LEAD() + Running Total")
q4.to_csv('sql_queries_output/q4_monthly_trend.csv', index=False)
q4.to_excel(excel_writer, sheet_name='Q4_Monthly_Trend', index=False)
print("  → Window: LAG() = previous month, LEAD() = next month preview, "
      "running SUM() OVER(ORDER BY) builds cumulative revenue timeline.")
print("  → Saved: q4_monthly_trend.csv\n")
 
 
# ─────────────────────────────────────────────────────────
# Q5 | Cancellation Analysis — 3-CTE Chain
# CTE concepts: base CTE → rate CTE → risk-flag CTE (chained)
# ─────────────────────────────────────────────────────────
q5 = run_query(cursor, """
    WITH sales_base AS (
        -- CTE 1: aggregate sales orders per category
        SELECT
            Category,
            COUNT(DISTINCT InvoiceNo)  AS SalesOrders,
            ROUND(SUM(Revenue), 2)     AS SalesRevenue
        FROM fact_transactions
        GROUP BY Category
    ),
    cancel_base AS (
        -- CTE 2: aggregate cancellation orders per category
        SELECT
            Category,
            COUNT(*)                   AS CancelOrders,
            ROUND(SUM(ABS(Quantity) * UnitPrice), 2) AS CancelledValue
        FROM fact_cancellations
        GROUP BY Category
    ),
    cancel_rate AS (
        -- CTE 3: join both and compute rate + risk flag
        SELECT
            s.Category,
            s.SalesOrders,
            s.SalesRevenue,
            COALESCE(c.CancelOrders, 0)    AS CancelOrders,
            COALESCE(c.CancelledValue, 0)  AS CancelledValue,
            ROUND(COALESCE(c.CancelOrders, 0) * 100.0
                  / s.SalesOrders, 2)      AS CancelRate_Pct,
            CASE
                WHEN ROUND(COALESCE(c.CancelOrders, 0) * 100.0
                           / s.SalesOrders, 2) > 2.0
                THEN 'High Risk'
                WHEN ROUND(COALESCE(c.CancelOrders, 0) * 100.0
                           / s.SalesOrders, 2) > 1.5
                THEN 'Medium Risk'
                ELSE 'Low Risk'
            END                            AS RiskFlag
        FROM sales_base s
        LEFT JOIN cancel_base c ON s.Category = c.Category
    )
    SELECT *,
        RANK() OVER (ORDER BY CancelRate_Pct DESC) AS CancelRank
    FROM cancel_rate
    ORDER BY CancelRate_Pct DESC;
""", "Q5 | Cancellation Risk Analysis — 3-CTE Chain + RANK()")
q5.to_csv('sql_queries_output/q5_cancellation_risk.csv', index=False)
q5.to_excel(excel_writer, sheet_name='Q5_Cancellation_Risk', index=False)
print("  → CTE chain: CTE1 (sales) → CTE2 (cancels) → CTE3 (join + rate + flag). "
      "Each CTE builds on the previous — cleaner than nested subqueries.")
print("  → Saved: q5_cancellation_risk.csv\n")
 
 
# ─────────────────────────────────────────────────────────
# Q6 | Customer Segments + Rank within Segment
# CTE + CASE WHEN + RANK() OVER(PARTITION BY segment)
# ─────────────────────────────────────────────────────────
q6 = run_query(cursor, """
    WITH customer_stats AS (
        -- CTE 1: per-customer aggregation
        SELECT
            CustomerID,
            COUNT(DISTINCT InvoiceNo)   AS TotalOrders,
            ROUND(SUM(Revenue), 2)      AS TotalRevenue,
            ROUND(AVG(Revenue), 2)      AS AvgOrderValue,
            MIN(InvoiceDate)            AS FirstPurchase,
            MAX(InvoiceDate)            AS LastPurchase
        FROM fact_transactions
        WHERE CustomerID != 'Guest'
        GROUP BY CustomerID
    ),
    segmented AS (
        -- CTE 2: assign loyalty segment using CASE WHEN
        SELECT *,
            CASE
                WHEN TotalOrders = 1  THEN '1_One-Time'
                WHEN TotalOrders <= 3 THEN '2_Occasional'
                WHEN TotalOrders <= 7 THEN '3_Regular'
                ELSE                       '4_Loyal'
            END AS Segment
        FROM customer_stats
    )
    SELECT
        CustomerID,
        Segment,
        TotalOrders,
        TotalRevenue,
        AvgOrderValue,
        FirstPurchase,
        LastPurchase,
        RANK() OVER (
            PARTITION BY Segment
            ORDER BY TotalRevenue DESC
        )                              AS RankInSegment,
        ROUND(TotalRevenue * 100.0
              / SUM(TotalRevenue) OVER (), 2) AS RevenueShare_Pct
    FROM segmented
    ORDER BY Segment, RankInSegment
    LIMIT 30;
""", "Q6 | Customer Segments + RANK() PARTITION BY Segment")
q6.to_csv('sql_queries_output/q6_customer_segments.csv', index=False)
q6.to_excel(excel_writer, sheet_name='Q6_Customer_Segments', index=False)
print("  → Window: RANK() PARTITION BY Segment ranks each customer WITHIN "
      "their segment — shows top spender per loyalty tier.")
print("  → Saved: q6_customer_segments.csv\n")
 
 
# ─────────────────────────────────────────────────────────
# Q7 | First & Last Purchase per Customer — FIRST_VALUE() + LAST_VALUE()
# Window functions: FIRST_VALUE(), LAST_VALUE() OVER(PARTITION BY CustomerID)
# ─────────────────────────────────────────────────────────
q7 = run_query(cursor, """
    WITH customer_journey AS (
        SELECT
            CustomerID,
            InvoiceNo,
            InvoiceDate,
            ROUND(Revenue, 2) AS Revenue,
            Category,
            FIRST_VALUE(InvoiceDate) OVER (
                PARTITION BY CustomerID
                ORDER BY InvoiceDate
                ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
            )                        AS FirstPurchaseDate,
            LAST_VALUE(InvoiceDate) OVER (
                PARTITION BY CustomerID
                ORDER BY InvoiceDate
                ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
            )                        AS LastPurchaseDate,
            FIRST_VALUE(Category) OVER (
                PARTITION BY CustomerID
                ORDER BY InvoiceDate
                ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
            )                        AS FirstCategory,
            LAST_VALUE(Category) OVER (
                PARTITION BY CustomerID
                ORDER BY InvoiceDate
                ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
            )                        AS LastCategory
        FROM fact_transactions
        WHERE CustomerID != 'Guest'
    ),
    summary AS (
        SELECT
            CustomerID,
            FirstPurchaseDate,
            LastPurchaseDate,
            FirstCategory,
            LastCategory,
            DATEDIFF(LastPurchaseDate, FirstPurchaseDate) AS DaysBetween,
            COUNT(DISTINCT InvoiceNo)                     AS TotalOrders,
            ROUND(SUM(Revenue), 2)                        AS TotalRevenue
        FROM customer_journey
        GROUP BY CustomerID, FirstPurchaseDate, LastPurchaseDate,
                 FirstCategory, LastCategory
    )
    SELECT *
    FROM summary
    WHERE TotalOrders > 1
    ORDER BY TotalRevenue DESC
    LIMIT 20;
""", "Q7 | Customer Journey — FIRST_VALUE() + LAST_VALUE()")
q7.to_csv('sql_queries_output/q7_customer_journey.csv', index=False)
q7.to_excel(excel_writer, sheet_name='Q7_Customer_Journey', index=False)
print("  → Window: FIRST_VALUE() & LAST_VALUE() with UNBOUNDED frame "
      "captures each customer's first and last purchase date & category.")
print("  → Saved: q7_customer_journey.csv\n")
 
 
# ─────────────────────────────────────────────────────────
# Q8 | 3-Month Rolling Average Revenue — AVG() OVER(ROWS BETWEEN)
# Window function: AVG() OVER(ORDER BY ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
# ─────────────────────────────────────────────────────────
q8 = run_query(cursor, """
    WITH monthly AS (
        SELECT
            YearMonth,
            ROUND(SUM(Revenue), 2)     AS MonthlyRevenue,
            COUNT(DISTINCT InvoiceNo)  AS TotalOrders
        FROM fact_transactions
        GROUP BY YearMonth
    )
    SELECT
        YearMonth,
        TotalOrders,
        MonthlyRevenue,
        ROUND(AVG(MonthlyRevenue) OVER (
            ORDER BY YearMonth
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2)                          AS Rolling3M_Avg,
        ROUND(AVG(MonthlyRevenue) OVER (
            ORDER BY YearMonth
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ), 2)                          AS Rolling6M_Avg,
        ROUND(MonthlyRevenue
              - AVG(MonthlyRevenue) OVER (
                    ORDER BY YearMonth
                    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
              ), 2)                    AS Deviation_from_3M_Avg
    FROM monthly
    ORDER BY YearMonth;
""", "Q8 | Rolling Averages — AVG() OVER(ROWS BETWEEN)")
q8.to_csv('sql_queries_output/q8_rolling_averages.csv', index=False)
q8.to_excel(excel_writer, sheet_name='Q8_Rolling_Averages', index=False)
print("  → Window: ROWS BETWEEN 2 PRECEDING AND CURRENT ROW = 3-month "
      "rolling window. Smooths seasonal spikes to reveal true trend.")
print("  → Saved: q8_rolling_averages.csv\n")
 
 
# ─────────────────────────────────────────────────────────
# Q9 | Pareto Analysis — Cumulative Revenue % by Customer
# CTE + SUM() OVER(ORDER BY) + cumulative % for 80/20 rule
# ─────────────────────────────────────────────────────────
q9 = run_query(cursor, """
    WITH customer_rev AS (
        -- CTE 1: total revenue per customer
        SELECT
            CustomerID,
            ROUND(SUM(Revenue), 2) AS TotalRevenue
        FROM fact_transactions
        WHERE CustomerID != 'Guest'
        GROUP BY CustomerID
    ),
    ranked AS (
        -- CTE 2: rank and compute cumulative revenue
        SELECT
            CustomerID,
            TotalRevenue,
            RANK() OVER (ORDER BY TotalRevenue DESC)   AS RevenueRank,
            ROUND(SUM(TotalRevenue) OVER (
                ORDER BY TotalRevenue DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ), 2)                                      AS CumulativeRevenue,
            ROUND(SUM(TotalRevenue) OVER (
                ORDER BY TotalRevenue DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) * 100.0 / SUM(TotalRevenue) OVER (), 2) AS CumulativePct
        FROM customer_rev
    )
    SELECT
        RevenueRank,
        CustomerID,
        TotalRevenue,
        CumulativeRevenue,
        CumulativePct,
        CASE
            WHEN CumulativePct <= 80 THEN 'Top 80% Revenue'
            ELSE 'Bottom 20% Revenue'
        END AS ParetoSegment
    FROM ranked
    ORDER BY RevenueRank
    LIMIT 30;
""", "Q9 | Pareto Analysis — Cumulative Revenue % (80/20 Rule)")
q9.to_csv('sql_queries_output/q9_pareto_analysis.csv', index=False)
q9.to_excel(excel_writer, sheet_name='Q9_Pareto_Analysis', index=False)
print("  → CTE chain + running SUM() OVER() builds cumulative % — "
      "identifies exactly which customers generate the top 80% of revenue.")
print("  → Saved: q9_pareto_analysis.csv\n")
 
 
# ─────────────────────────────────────────────────────────
# Q10 | Customer Revenue Quartiles — NTILE(4)
# Window function: NTILE(4) splits customers into 4 equal revenue buckets
# ─────────────────────────────────────────────────────────
q10 = run_query(cursor, """
    WITH customer_metrics AS (
        SELECT
            CustomerID,
            COUNT(DISTINCT InvoiceNo)   AS TotalOrders,
            ROUND(SUM(Revenue), 2)      AS TotalRevenue,
            ROUND(AVG(Revenue), 2)      AS AvgOrderValue,
            COUNT(DISTINCT Category)    AS CategoriesBought
        FROM fact_transactions
        WHERE CustomerID != 'Guest'
          AND IsBulkOrder  = 0
        GROUP BY CustomerID
    ),
    quartiled AS (
        SELECT *,
            NTILE(4) OVER (ORDER BY TotalRevenue DESC) AS RevenueQuartile
        FROM customer_metrics
    )
    SELECT
        RevenueQuartile,
        CASE RevenueQuartile
            WHEN 1 THEN 'Q1 — Top 25% (Premium)'
            WHEN 2 THEN 'Q2 — Upper Mid (Growth)'
            WHEN 3 THEN 'Q3 — Lower Mid (Nurture)'
            WHEN 4 THEN 'Q4 — Bottom 25% (At Risk)'
        END                                            AS QuartileLabel,
        COUNT(*)                                       AS NumCustomers,
        ROUND(MIN(TotalRevenue), 2)                    AS MinRevenue,
        ROUND(MAX(TotalRevenue), 2)                    AS MaxRevenue,
        ROUND(AVG(TotalRevenue), 2)                    AS AvgRevenue,
        ROUND(SUM(TotalRevenue), 2)                    AS TotalSegmentRevenue,
        ROUND(SUM(TotalRevenue) * 100.0
              / SUM(SUM(TotalRevenue)) OVER (), 2)     AS RevenueShare_Pct
    FROM quartiled
    GROUP BY RevenueQuartile
    ORDER BY RevenueQuartile;
""", "Q10 | Customer Revenue Quartiles — NTILE(4)")
q10.to_csv('sql_queries_output/q10_revenue_quartiles.csv', index=False)
q10.to_excel(excel_writer, sheet_name='Q10_Revenue_Quartiles', index=False)
print("  → Window: NTILE(4) splits all customers into 4 equal-sized buckets "
      "by revenue — Q1=Premium, Q4=At-Risk. Direct input for churn model.")
print("  → Saved: q10_revenue_quartiles.csv\n")
 
 
# ── Save Excel workbook ────────────────────────────────────
excel_writer.close()
print(f"  ✓ Combined Excel saved: {excel_path}")
print("    (10 sheets — one per query)")


════════════════════════════════════════════════════════════
  Q1 | Revenue by Country — RANK() + Cumulative Revenue
════════════════════════════════════════════════════════════
       Country  TotalOrders TotalRevenue  RevenueRank RevenueShare_Pct CumulativeRevenue
United Kingdom        21143  15561967.90            1            77.67       15561967.90
       Germany         1185    806481.48            2             4.03       16368449.38
        France         1021    766413.19            3             3.83       17134862.57
       Ireland          882    614737.52            4             3.07       17749600.09
   Netherlands          694    497228.36            5             2.48       18246828.45
         Other          592    467878.13            6             2.34       18714706.58
         Spain          577    426114.48            7             2.13       19140821.06
     Australia          480    347583.57            8             1.73       19488404.63
       Belgium      

In [17]:
cursor.close()
conn.close()

In [31]:
print("Current directory:", os.getcwd())
print("Folder exists:", os.path.exists('sql_queries_output'))

Current directory: C:\Users\HP\E-commerce Analytics
Folder exists: False


In [32]:
os.chdir(r'C:\Users\HP\E-commerce Analytics')
os.makedirs('sql_queries_output', exist_ok=True)
print("Current directory:", os.getcwd())
print("Folder exists:", os.path.exists('sql_queries_output'))

Current directory: C:\Users\HP\E-commerce Analytics
Folder exists: True
